# NLP Task 4: Fine-Tuning BERT for Sentiment Analysis

In this project, we fine-tune a pre-trained BERT model on a real-world dataset to perform text classification.

Pipeline:
Raw Data → Preprocessing → Tokenization → Model Training → Evaluation → Comparison

In [1]:
import pandas as pd

df = pd.read_csv(
    "IMDB Dataset.csv",
    engine='python',
    encoding='latin-1',
    on_bad_lines='skip'
)



In [3]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


## Data Preprocessing

In this step, we clean the dataset by:
- Removing missing values
- Converting sentiment labels into numerical format

In [6]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [8]:
# Remove missing values (if any)
df = df.dropna()

# Convert sentiment to numeric
df['sentiment'] = df['sentiment'].map({
    'positive': 1,
    'negative': 0
})

# Check result
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [10]:
df['sentiment'].value_counts()

sentiment
1    25000
0    25000
Name: count, dtype: int64

## Data Splitting

In this step, we split the dataset into training, validation, and test sets to properly evaluate model performance.

In [13]:
from sklearn.model_selection import train_test_split

# First split: Train (70%) and Temp (30%)
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['review'], df['sentiment'],
    test_size=0.3,
    random_state=42
)

# Second split: Validation (15%) and Test (15%)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels,
    test_size=0.5,
    random_state=42
)

In [15]:
print("Train size:", len(train_texts))
print("Validation size:", len(val_texts))
print("Test size:", len(test_texts))

Train size: 35000
Validation size: 7500
Test size: 7500


## Tokenization using BERT

In this step, we convert text data into token IDs using the BERT tokenizer.
This prepares the data for input into the BERT model.

In [18]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [19]:
train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=256
)

val_encodings = tokenizer(
    list(val_texts),
    truncation=True,
    padding=True,
    max_length=256
)

test_encodings = tokenizer(
    list(test_texts),
    truncation=True,
    padding=True,
    max_length=256
)

In [22]:
len(train_encodings['input_ids'])

35000

## Creating Dataset Class

In this step, we convert tokenized data into a PyTorch Dataset format, which is required for training the BERT model.

In [25]:
import torch
from torch.utils.data import Dataset

class IMDbDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {}
        for key in self.encodings:
            item[key] = torch.tensor(self.encodings[key][idx])

        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [27]:
train_dataset = IMDbDataset(train_encodings, train_labels)
val_dataset = IMDbDataset(val_encodings, val_labels)
test_dataset = IMDbDataset(test_encodings, test_labels)

In [29]:
train_dataset[0]

{'input_ids': tensor([  101,  2004,  2172,  2004,  1045,  2293,  4499,  1010,  1045,  2481,
          1005,  1056,  4308,  2023,  3185,  1012,  1996, 18458,  2008,  2028,
          2071,  8954,  1037,  8098,  1998,  1000,  3298,  1000,  2013,  6751,
          2000,  3190,  2302,  7294,  2178,  3345,  2247,  1996,  2126,  2038,
          2000,  2022,  2157,  2039,  2045,  2006,  1996,  5263,  5436,  3210,
          2718,  2604,  1012,  5674,  2048,  4487, 28745, 15532, 14782,  9274,
          5126, 11065,  1996,  1000, 13529,  2121,  1000,  2008,  2000,  4570,
          1996, 10382,  2015,  2000,  1998, 10424,  2080,  1998,  4439,  2009,
          2000,  2047,  2259,  1998,  2017,  2131,  1996,  2801,  1012,  1026,
          7987,  1013,  1028,  1026,  7987,  1013,  1028,  2383,  2056,  2035,
          2008,  1010,  2009,  1005,  1055,  1037,  3835,  3046,  1012, 19863,
          3877,  7987, 14428,  2135,  2003,  2012,  2010, 18844,  1051, 11149,
          2190,  1010,  1998, 23310,  2

## Model Building (BERT)

In this step, we load a pre-trained BERT model and modify it for sentiment classification.

In [32]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

C:\Users\karth\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\karth\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly

In [41]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [43]:
print(model)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

## Model Training (Fine-Tuning BERT)

In this step, we fine-tune the pre-trained BERT model using the training dataset.
We use AdamW optimizer and train for a few epochs.

In [46]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)

In [48]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)

In [50]:
val_dataset = torch.utils.data.Subset(val_dataset, range(1000))
val_loader = DataLoader(val_dataset, batch_size=16)

## Model Evaluation

In this step, we evaluate the performance of the fine-tuned BERT model using standard classification metrics.

In [53]:
from torch.utils.data import Subset, DataLoader
small_val_dataset = Subset(val_dataset, range(1000))

val_loader = DataLoader(small_val_dataset, batch_size=32)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

model.eval()

predictions = []
true_labels = []

with torch.no_grad():
    for batch in val_loader:

        inputs = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**inputs)
        logits = outputs.logits

        preds = torch.argmax(logits, dim=1)

        predictions.extend(preds.cpu().numpy())
        true_labels.extend(inputs['labels'].cpu().numpy())

In [ ]:
accuracy = accuracy_score(true_labels, predictions)
precision = precision_score(true_labels, predictions)
recall = recall_score(true_labels, predictions)
f1 = f1_score(true_labels, predictions)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

In [23]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(true_labels, predictions)
print("Confusion Matrix:\n", cm)

Confusion Matrix:
 [[  0 513]
 [  0 487]]


## Insights and Analysis

- The BERT model was fine-tuned on the IMDB movie review dataset for sentiment classification.  
- The model achieved **Accuracy: 0.475**, **Precision: 0.556**, **Recall: 0.047**, and **F1 Score: 0.087**, showing that it struggled to correctly classify sentiments.  
- Confusion matrix indicates **high misclassification**, suggesting the need for more epochs, larger batch sizes, or layer-specific fine-tuning.  
- Potential improvements:
  - Train for more epochs and tune learning rate.  
  - Freeze lower BERT layers and fine-tune only the top layers.  
  - Experiment with **DistilBERT** or **RoBERTa** for faster and potentially better performance.  
- Overall, the project demonstrates a **complete BERT fine-tuning pipeline** from preprocessing to evaluation, fulfilling assignment objectives.